# Random Forest From Scratch

Implemented using **NumPy** — `sklearn` is only used to load the dataset.

In this notebook we implement **Random Forest from scratch**, building directly on our Decision Tree.

We cover:
- What Random Forest is and why it works
- **Bootstrap Sampling** — training each tree on a random subset of the data
- **Feature Subsampling** — each split considers only a random subset of features
- **Ensemble Voting** — combining trees to make a final prediction
- A clean, reusable `RandomForest` class with `fit` and `predict`
- Our own **`train_test_split`** — no sklearn utilities

Resources that helped me build this:

Video — Intuition & Implementation: https://youtu.be/v6VJ2RO66Ag?si=hmbFtXUYqWkRHHlT

## What is Random Forest?

A **Random Forest** is an **ensemble** of Decision Trees. Instead of relying on a single tree (which tends to overfit), we build many trees and let them vote.

The key insight is **variance reduction through randomness**. Each tree is made deliberately different from the others via two mechanisms:

### 1. Bootstrap Sampling (Bagging)
Each tree is trained on a **bootstrap sample** — a random sample of the training data drawn **with replacement**, the same size as the original dataset. On average, each bootstrap sample contains ~63% unique samples; the rest are duplicates.

### 2. Feature Subsampling
At **every split**, each tree only considers a **random subset of features** (typically $\sqrt{n\_features}$) rather than all features. This decorrelates the trees — they don't all pick the same dominant feature every time.

### 3. Majority Vote
At prediction time, each tree votes for a class. The forest returns the **majority vote** across all trees.

Together, these two sources of randomness produce trees that are:
- **Diverse** — they make different errors
- **Averaged** — errors cancel out across the ensemble

This is why Random Forests consistently outperform single Decision Trees.

Let's start coding 🤓.

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.datasets import load_breast_cancer  # only used to load the dataset

## 2. Load & Explore Data

We use the **Breast Cancer Wisconsin** dataset — the same one from our KNN and Naive Bayes notebooks — so you can directly compare accuracy across algorithms.

In [ ]:
bc = load_breast_cancer()

X = bc.data                             # shape (569, 30)
y = np.where(bc.target == 1, "B", "M") # 1 → Benign, 0 → Malignant

print(f"Dataset shape : {X.shape}")
print(f"Classes       : {np.unique(y)}")
print(f"Class counts  : B={np.sum(y == 'B')}, M={np.sum(y == 'M')}")

## 3. Train / Test Split — From Scratch

In [ ]:
def train_test_split(X, y, test_size=0.2, random_state=None):
    if random_state is not None:
        np.random.seed(random_state)
    m = len(y)
    indices = np.random.permutation(m)
    split = int(m * (1 - test_size))
    train_idx, test_idx = indices[:split], indices[split:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train size : {len(y_train)}")
print(f"Test  size : {len(y_test)}")

##4. The Decision Tree — Our Building Block

Random Forest is built on top of Decision Trees. We reuse our implementation from the Decision Tree notebook with one addition: the `max_features` parameter controls how many features each split is allowed to consider.

This is the core of feature subsampling — each tree sees only a random subset of features at every node.

In [ ]:
# ── Node ────────────────────────────────────────────────────────────────────────
class Node:
    def __init__(self, feature_idx=None, threshold=None, info_gain=None,
                 left=None, right=None, value=None):
        self.feature_idx = feature_idx
        self.threshold   = threshold
        self.info_gain   = info_gain
        self.left        = left
        self.right       = right
        self.value       = value

    @property
    def is_leaf(self):
        return self.value is not None


# ── Utility functions ────────────────────────────────────────────────────────────
def entropy(y):
    if len(y) == 0:
        return 0.0
    _, counts = np.unique(y, return_counts=True)
    p = counts / len(y)
    return -np.sum(p * np.log2(p + 1e-12))


def information_gain(parent_y, left_y, right_y):
    lw = len(left_y)  / len(parent_y)
    rw = len(right_y) / len(parent_y)
    return entropy(parent_y) - (lw * entropy(left_y) + rw * entropy(right_y))


# ── Decision Tree ────────────────────────────────────────────────────────────────
class DecisionTree:
    def __init__(self, min_samples_split=2, max_depth=10, max_features=None):
        self.min_samples_split = min_samples_split
        self.max_depth         = max_depth
        self.max_features      = max_features   # None → use all features
        self.root              = None

    def _split(self, dataset, feature_idx, threshold):
        mask = dataset[:, feature_idx] <= threshold
        return dataset[mask], dataset[~mask]

    def _best_split(self, dataset, n_features):
        # ── Feature subsampling ──────────────────────────────────────────────────
        k = self.max_features if self.max_features else n_features
        feature_indices = np.random.choice(n_features, k, replace=False)

        best = {"feature_idx": None, "threshold": None,
                "info_gain": -1, "left_dataset": None, "right_dataset": None}

        for fi in feature_indices:
            for threshold in np.unique(dataset[:, fi]):
                left_ds, right_ds = self._split(dataset, fi, threshold)
                if len(left_ds) == 0 or len(right_ds) == 0:
                    continue
                ig = information_gain(dataset[:, -1], left_ds[:, -1], right_ds[:, -1])
                if ig > best["info_gain"]:
                    best.update(feature_idx=fi, threshold=threshold,
                                info_gain=ig, left_dataset=left_ds, right_dataset=right_ds)
        return best

    def _build_tree(self, dataset, depth=0):
        X, y = dataset[:, :-1], dataset[:, -1]
        n_samples, n_features = X.shape

        if n_samples < self.min_samples_split or depth >= self.max_depth:
            return Node(value=Counter(y).most_common(1)[0][0])

        best = self._best_split(dataset, n_features)
        if best["info_gain"] <= 0:
            return Node(value=Counter(y).most_common(1)[0][0])

        return Node(
            feature_idx=best["feature_idx"],
            threshold=best["threshold"],
            info_gain=best["info_gain"],
            left=self._build_tree(best["left_dataset"],  depth + 1),
            right=self._build_tree(best["right_dataset"], depth + 1),
        )

    def fit(self, X, y):
        dataset   = np.concatenate([X, y.reshape(-1, 1)], axis=1)
        self.root = self._build_tree(dataset)
        return self

    def predict(self, X):
        return np.array([self._traverse(x, self.root) for x in X])

    def _traverse(self, x, node):
        if node.is_leaf:
            return node.value
        if x[node.feature_idx] <= node.threshold:
            return self._traverse(x, node.left)
        return self._traverse(x, node.right)

## 5. The RandomForest Class

**`fit`** — build `n_trees` Decision Trees, each trained on a different bootstrap sample with feature subsampling enabled.

**`predict`** — collect a vote from every tree and return the majority class.

Bootstrap sampling is done with `np.random.choice(..., replace=True)`.

In [ ]:
class RandomForest:
    def __init__(self, n_trees=100, max_depth=10, min_samples_split=2,
                 max_features=None, random_state=None):
        self.n_trees          = n_trees
        self.max_depth        = max_depth
        self.min_samples_split = min_samples_split
        self.max_features     = max_features   # defaults to sqrt(n_features) in fit()
        self.random_state     = random_state
        self.trees_           = []

    def _bootstrap(self, X, y):
        """Draw a bootstrap sample (same size as X, with replacement)."""
        m = len(y)
        indices = np.random.choice(m, m, replace=True)
        return X[indices], y[indices]

    def fit(self, X, y):
        """Train n_trees Decision Trees, each on a bootstrap sample."""
        if self.random_state is not None:
            np.random.seed(self.random_state)

        # Default: sqrt(n_features) features per split
        max_features = self.max_features or int(np.sqrt(X.shape[1]))

        self.trees_ = []
        for _ in range(self.n_trees):
            X_boot, y_boot = self._bootstrap(X, y)
            tree = DecisionTree(
                min_samples_split=self.min_samples_split,
                max_depth=self.max_depth,
                max_features=max_features,
            )
            tree.fit(X_boot, y_boot)
            self.trees_.append(tree)

        return self

    def predict(self, X):
        """Majority vote across all trees."""
        # shape (n_trees, m)
        votes = np.array([tree.predict(X) for tree in self.trees_])
        # majority vote for each sample
        return np.array([
            Counter(votes[:, i]).most_common(1)[0][0]
            for i in range(X.shape[0])
        ])

## 6. Train & Evaluate

We compare a **single Decision Tree** against the **Random Forest** to show the ensemble benefit.

In [ ]:
# Single tree baseline
single_tree = DecisionTree(max_depth=10)
single_tree.fit(X_train, y_train)
tree_preds   = single_tree.predict(X_test)
tree_acc     = np.mean(tree_preds == y_test) * 100

# Random Forest
rf = RandomForest(n_trees=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
rf_acc   = np.mean(rf_preds == y_test) * 100

print(f"Single Decision Tree accuracy : {tree_acc:.2f}%")
print(f"Random Forest accuracy        : {rf_acc:.2f}%")

## 7. Accuracy vs Number of Trees

As we add more trees the ensemble gets more stable. Let's see where it plateaus.

In [ ]:
tree_counts = [1, 5, 10, 20, 50, 100]
accuracies  = []

for n in tree_counts:
    rf_n = RandomForest(n_trees=n, max_depth=10, random_state=42)
    rf_n.fit(X_train, y_train)
    acc = np.mean(rf_n.predict(X_test) == y_test) * 100
    accuracies.append(acc)
    print(f"n_trees={n:4d} | Accuracy: {acc:.2f}%")

plt.figure(figsize=(7, 4))
plt.plot(tree_counts, accuracies, marker="o", color="steelblue")
plt.xlabel("Number of Trees")
plt.ylabel("Test Accuracy (%)")
plt.title("Random Forest — Accuracy vs Number of Trees")
plt.grid(True, alpha=0.3)
plt.show()

## 8. Feature Importance

A nice property of tree-based models is that we can measure **feature importance** — how much each feature contributed to the splits across all trees.

We use **mean decrease in impurity (MDI)**: for each feature, we sum the information gain from every split on that feature, weighted by the number of samples that reached that node, then average across all trees.

In [ ]:
def tree_feature_importance(tree, n_features):
    """Recursively collect weighted information gain per feature."""
    importance = np.zeros(n_features)

    def _recurse(node, n_samples):
        if node is None or node.is_leaf:
            return
        importance[node.feature_idx] += node.info_gain * n_samples
        # approximate child sizes (we don't store them, so split 50/50 as proxy)
        _recurse(node.left,  n_samples // 2)
        _recurse(node.right, n_samples // 2)

    _recurse(tree.root, len(X_train))
    return importance


n_features   = X_train.shape[1]
importances  = np.mean(
    [tree_feature_importance(t, n_features) for t in rf.trees_], axis=0
)
importances /= importances.sum()   # normalize to sum to 1

# Top 10 features
top_idx = np.argsort(importances)[::-1][:10]

plt.figure(figsize=(9, 4))
plt.bar(range(10), importances[top_idx], color="steelblue")
plt.xticks(range(10), bc.feature_names[top_idx], rotation=30, ha="right")
plt.ylabel("Relative Importance")
plt.title("Random Forest — Top 10 Feature Importances")
plt.tight_layout()
plt.show()